# RADPS Stack Validation Notebook

This notebook validates that the full RADPS infrastructure stack is working correctly on your local k3s cluster. It tests:

1. **Dask Kubernetes Operator** — dynamically creating a Dask cluster
2. **Dask distributed computing** — running tasks across workers
3. **Prefect integration** — running flows with DaskTaskRunner

No external data is required — all workloads use synthetic data.

## 1. Imports

In [ ]:
from dask_kubernetes.operator import KubeCluster, make_cluster_spec
from dask.distributed import Client
import dask.array as da
import dask
import numpy as np
import time
import os

## 2. Create a Dask cluster via the Kubernetes Operator

This creates a `DaskCluster` custom resource. The Dask Operator sees it and provisions a scheduler pod and worker pods.

In [ ]:
username = os.environ["JUPYTERHUB_USER"]

spec = make_cluster_spec(
    name=f"test-cluster-{username}",
    image="ghcr.io/casangi/radps-dask-worker",
    worker_command=[
        "dask-worker",
        "--nworkers", "2",
        "--nthreads", "1",
        "--memory-limit", "1G",
    ],
    scheduler_service_type="NodePort"
)

# Use local arm64 image instead of pulling from ghcr.io
for component in ["worker", "scheduler"]:
    containers = spec["spec"][component]["spec"]["containers"]
    for container in containers:
        container["imagePullPolicy"] = "IfNotPresent"

cluster = KubeCluster(
    custom_cluster_spec=spec,
    namespace="radps-hub",
)

In [ ]:
cluster.scale(1)
cluster.wait_for_workers(1)
print(f"Cluster ready: 1 pod, 2 workers")

In [ ]:
client = Client(cluster)
print("Dask client connected.")
client

## 3. Test Dask with synthetic data

Run a distributed computation using dask arrays to verify the workers are functioning. This mirrors the kind of workload the pipeline performs (covariance on random visibility-like data).

In [ ]:
# Simulate a visibility-like dataset: baselines x time x channels
n_baselines, n_time, n_chan = 500, 100, 64
vis_data = da.random.normal(size=(n_baselines, n_time, n_chan), chunks=(100, 100, 64)) \
         + 1j * da.random.normal(size=(n_baselines, n_time, n_chan), chunks=(100, 100, 64))

print(f"Synthetic visibility data: {vis_data.shape}, {vis_data.nbytes / 1e6:.1f} MB")

# Compute the mean amplitude per channel (a typical calibration-like reduction)
mean_amplitude = da.abs(vis_data).mean(axis=(0, 1))
result = mean_amplitude.compute()

print(f"Mean amplitude per channel: shape={result.shape}, mean={result.mean():.4f}")
print("Dask distributed computation: OK")

## 4. Test dask.delayed workload

Run many small tasks via `dask.delayed` to verify task scheduling and collection across workers.

In [ ]:
def process_channel_chunk(channel_id):
    """Simulate processing a single channel chunk."""
    import numpy as np
    import time
    time.sleep(0.05)
    data = np.random.normal(size=(100, 100))
    return float(np.std(data))

n_chunks = 100
delayed_results = [dask.delayed(process_channel_chunk)(i) for i in range(n_chunks)]
results = dask.compute(*delayed_results)

print(f"Processed {n_chunks} channel chunks across workers")
print(f"Result std range: [{min(results):.4f}, {max(results):.4f}]")
print("dask.delayed workload: OK")

## 5. Test Prefect with DaskTaskRunner

Run a Prefect flow that distributes tasks across the Dask cluster. This validates the Prefect Server connection and the DaskTaskRunner integration.

In [ ]:
from prefect import flow, task
from prefect_dask.task_runners import DaskTaskRunner

@task
def image_single_spw(spw_id):
    """Simulate imaging a single spectral window."""
    import numpy as np
    import time
    time.sleep(0.5)
    image = np.random.normal(size=(256, 256))
    return {"spw": spw_id, "peak": float(np.max(np.abs(image)))}

task_runner = DaskTaskRunner(address=client.scheduler.address)

@flow(task_runner=task_runner)
def image_cube(n_spw):
    futures = [image_single_spw.submit(i) for i in range(n_spw)]
    return [f.result() for f in futures]

results = image_cube(n_spw=8)
for r in results:
    print(f"  SPW {r['spw']}: peak = {r['peak']:.4f}")
print("Prefect + DaskTaskRunner: OK")

## 6. Cleanup

Shut down the Dask client and cluster to release Kubernetes resources.

In [ ]:
client.close()
cluster.close()
print("Cluster shut down.")